# 🏀 March Madness Betting Analysis (2016-2025)

**Identifying profitable trends in spreads, totals, and upsets**

This notebook analyzes ~9 tournaments (2020 cancelled) to find actionable betting edges:
- ATS (Against the Spread) cover rates by seed matchup, round, and spread range
- Total points scoring distributions and over/under probabilities
- Upset patterns and Cinderella run frequency
- Flat-bet profitability simulations

⚠️ **Disclaimer**: This is for research/entertainment only. Past performance ≠ future results. Gamble responsibly.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from data_builder import build_dataset
from analysis import (
    spread_cover_by_seed_matchup, spread_cover_by_round, spread_cover_by_seed,
    spread_cover_by_spread_range, totals_by_round, totals_by_seed_matchup,
    totals_trend_by_year, totals_distribution, upset_rates_by_matchup,
    upset_rates_by_round, cinderella_analysis, flat_bet_simulation,
    strategy_summary, binomial_test_cover_rate, generate_key_findings
)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Setup complete.')

In [ ]:
# Build the dataset
df = build_dataset()
print(f"\nDataset: {len(df)} games across {df['year'].nunique()} tournaments")
print(f"Years: {sorted(df['year'].unique())}")
df.head()

---
## 1. KEY FINDINGS AT A GLANCE

In [ ]:
findings = generate_key_findings(df)
print("=" * 70)
print("KEY BETTING INSIGHTS")
print("=" * 70)
for i, f in enumerate(findings, 1):
    print(f"\n{i}. {f}")

---
## 2. SPREAD ANALYSIS (ATS)

In [ ]:
# ATS Cover Rate by Seed Matchup
matchup_ats = spread_cover_by_seed_matchup(df)
print("FAVORITE COVER RATE BY SEED MATCHUP")
print("(Lower fav_cover_pct = more value on the underdog)\n")
display(matchup_ats[['matchup', 'games', 'fav_cover_pct', 'dog_cover_pct', 
                      'upset_pct', 'avg_margin', 'dog_profit_per_100']]
        .sort_values('dog_cover_pct', ascending=False))

In [ ]:
# Visualize: Underdog cover rate by matchup
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of dog cover rate
top_matchups = matchup_ats[matchup_ats['games'] >= 5].sort_values('dog_cover_pct', ascending=True)
colors = ['#2ecc71' if x > 0.5 else '#e74c3c' for x in top_matchups['dog_cover_pct']]
axes[0].barh(top_matchups['matchup'], top_matchups['dog_cover_pct'], color=colors)
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.7, label='Break-even (50%)')
axes[0].axvline(x=0.524, color='orange', linestyle='--', alpha=0.7, label='Profitable (52.4%)')
axes[0].set_xlabel('Underdog Cover Rate')
axes[0].set_title('Underdog ATS Cover Rate by Matchup\n(Green = profitable at -110)')
axes[0].legend()

# Profitability chart
colors2 = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_matchups['dog_profit_per_100']]
axes[1].barh(top_matchups['matchup'], top_matchups['dog_profit_per_100'], color=colors2)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Profit per $100 bet')
axes[1].set_title('Underdog Flat Bet Profit by Matchup\n($100/game at -110)')

plt.tight_layout()
plt.savefig('data/spread_by_matchup.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ATS Cover Rate by Round
round_ats = spread_cover_by_round(df)
print("\nFAVORITE COVER RATE BY ROUND")
print("(Identifies which rounds favor underdogs)\n")
display(round_ats[['round_name', 'games', 'fav_cover_pct', 'dog_cover_pct', 'upset_pct', 'avg_margin']])

In [ ]:
# Visualize: Cover rate by round
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(round_ats))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], round_ats['fav_cover_pct'], width, 
               label='Favorite Covers', color='#3498db', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], round_ats['dog_cover_pct'], width,
               label='Underdog Covers', color='#e74c3c', alpha=0.8)

ax.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
ax.axhline(y=0.524, color='orange', linestyle='--', alpha=0.5, label='Profit threshold (52.4%)')
ax.set_xticks(x)
ax.set_xticklabels(round_ats['round_name'], rotation=30)
ax.set_ylabel('Cover Rate')
ax.set_title('ATS Cover Rate by Tournament Round')
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('data/spread_by_round.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ATS by Spread Range
spread_range = spread_cover_by_spread_range(df)
print("\nCOVER RATE BY ESTIMATED SPREAD RANGE")
print("(Are big favorites or small favorites better ATS?)\n")
display(spread_range)

In [ ]:
# Statistical test: Is the favorite cover rate significantly different from 50%?
print("\nSTATISTICAL TESTS")
print("="*50)

overall_test = binomial_test_cover_rate(df)
print(f"\nOverall favorite cover rate: {overall_test['fav_cover_pct']:.1%}")
print(f"  95% CI: ({overall_test['ci_low']:.1%}, {overall_test['ci_high']:.1%})")
print(f"  p-value: {overall_test['p_value']:.4f}")
print(f"  Significant at 5%: {overall_test['significant_at_05']}")

# Test first round specifically
r1_test = binomial_test_cover_rate(df, df['round_num'] == 1)
print(f"\nFirst Round favorite cover rate: {r1_test['fav_cover_pct']:.1%}")
print(f"  95% CI: ({r1_test['ci_low']:.1%}, {r1_test['ci_high']:.1%})")
print(f"  p-value: {r1_test['p_value']:.4f}")
print(f"  Significant at 5%: {r1_test['significant_at_05']}")

---
## 3. TOTALS ANALYSIS (Over/Under)

In [ ]:
# Total Points by Round
round_totals = totals_by_round(df)
print("TOTAL POINTS BY ROUND")
print("(Key for setting over/under expectations)\n")
display(round_totals)

In [ ]:
# Visualize: Scoring distribution by round
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
round_names = ['First Round', 'Second Round', 'Sweet 16', 'Elite 8', 'Final Four', 'Championship']

for idx, (ax, rnd) in enumerate(zip(axes.flat, range(1, 7))):
    round_data = df[df['round_num'] == rnd]['total_points']
    if len(round_data) > 0:
        ax.hist(round_data, bins=15, color=sns.color_palette('husl', 6)[idx], 
                alpha=0.7, edgecolor='black')
        ax.axvline(round_data.mean(), color='red', linestyle='--', 
                   label=f'Mean: {round_data.mean():.0f}')
        ax.axvline(round_data.median(), color='blue', linestyle='--',
                   label=f'Median: {round_data.median():.0f}')
        ax.set_title(f'{round_names[idx]} (n={len(round_data)})')
        ax.set_xlabel('Total Points')
        ax.legend(fontsize=8)

plt.suptitle('Total Points Distribution by Round', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/totals_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Over/Under probability table
print("\nPROBABILITY OF HITTING OVER/UNDER THRESHOLDS")
print("="*60)

for rnd in [None, 1, 3, 5]:
    dist = totals_distribution(df, rnd)
    label = "All Rounds" if rnd is None else {1: 'First Round', 3: 'Sweet 16', 5: 'Final Four'}[rnd]
    print(f"\n{label} (n={dist['count']}, mean={dist['mean']:.1f}, std={dist['std']:.1f})")
    print(f"  Percentiles: 10th={dist['percentiles']['10th']:.0f}, "
          f"25th={dist['percentiles']['25th']:.0f}, "
          f"50th={dist['percentiles']['50th']:.0f}, "
          f"75th={dist['percentiles']['75th']:.0f}, "
          f"90th={dist['percentiles']['90th']:.0f}")
    print(f"  Over probabilities:")
    for threshold in [130, 140, 150, 160]:
        pct = dist['over_probability'][threshold]
        print(f"    Over {threshold}: {pct:.1%}")

In [ ]:
# Totals trend over years
year_totals = totals_trend_by_year(df)
print("\nSCORING TRENDS BY YEAR")
display(year_totals)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(year_totals['year'], year_totals['avg_total'], 'o-', linewidth=2, markersize=8)
ax.fill_between(year_totals['year'], 
                year_totals['avg_total'] - 5, 
                year_totals['avg_total'] + 5, alpha=0.2)
ax.set_xlabel('Year')
ax.set_ylabel('Average Total Points')
ax.set_title('Average Total Points per Game by Year')
ax.axhline(y=df['total_points'].mean(), color='red', linestyle='--', 
           label=f'Overall avg: {df["total_points"].mean():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('data/totals_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. UPSET ANALYSIS

In [ ]:
# Upset rates by matchup
upset_matchup = upset_rates_by_matchup(df)
print("UPSET RATES BY SEED MATCHUP")
print("(Higher = more value on underdog moneyline)\n")
display(upset_matchup[upset_matchup['games'] >= 5].sort_values('upset_pct', ascending=False))

In [ ]:
# Visualize upset rates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# First round matchups only
r1_upsets = df[df['round_num'] == 1].groupby('matchup').agg(
    games=('upset', 'count'),
    upsets=('upset', 'sum')
).reset_index()
r1_upsets['upset_pct'] = r1_upsets['upsets'] / r1_upsets['games']
r1_upsets = r1_upsets[r1_upsets['games'] >= 3].sort_values('upset_pct', ascending=True)

colors = plt.cm.RdYlGn(r1_upsets['upset_pct'] / r1_upsets['upset_pct'].max())
axes[0].barh(r1_upsets['matchup'], r1_upsets['upset_pct'], color=colors)
axes[0].set_xlabel('Upset Rate')
axes[0].set_title('First Round Upset Rate by Matchup')
for i, (_, row) in enumerate(r1_upsets.iterrows()):
    axes[0].text(row['upset_pct'] + 0.01, i, 
                 f"{row['upsets']:.0f}/{row['games']:.0f}", va='center', fontsize=9)

# Upset rate by round
round_upsets = upset_rates_by_round(df)
axes[1].bar(round_upsets['round_name'], round_upsets['upset_pct'], 
            color=sns.color_palette('husl', len(round_upsets)))
axes[1].set_ylabel('Upset Rate')
axes[1].set_title('Upset Rate by Round')
axes[1].tick_params(axis='x', rotation=30)
for i, (_, row) in enumerate(round_upsets.iterrows()):
    axes[1].text(i, row['upset_pct'] + 0.01, f"{row['upset_pct']:.0%}", 
                 ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('data/upset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cinderella runs (seed 9+)
cinderellas = cinderella_analysis(df, min_seed=9)
print("\nNOTABLE CINDERELLA RUNS (Seed 9+)")
print("="*60)
display(cinderellas.head(20))

---
## 5. PROFITABILITY SIMULATION

In [ ]:
# Simulate flat betting on underdogs ATS
print("FLAT BET SIMULATION: $100/game on Underdogs ATS (-110)")
print("="*60)

dog_sim = flat_bet_simulation(df, bet_type='underdog_spread')
dog_summary = strategy_summary(dog_sim)

print(f"Total bets: {dog_summary['total_bets']}")
print(f"Record: {dog_summary['wins']}-{dog_summary['losses']}")
print(f"Win %: {dog_summary['win_pct']:.1%}")
print(f"Total profit: ${dog_summary['total_profit']:+,.0f}")
print(f"Avg profit/bet: ${dog_summary['avg_profit_per_bet']:+,.2f}")
print(f"\nProfit by year:")
for year, profit in sorted(dog_summary['profit_by_year'].items()):
    print(f"  {year}: ${profit:+,.0f}")

print(f"\nProfit by round:")
for rnd, profit in dog_summary['profit_by_round'].items():
    print(f"  {rnd}: ${profit:+,.0f}")

In [ ]:
# Simulate underdog moneyline bets
print("\nFLAT BET SIMULATION: $100/game on Underdog Moneyline")
print("="*60)

ml_sim = flat_bet_simulation(df, bet_type='underdog_ml')
ml_summary = strategy_summary(ml_sim)

print(f"Total bets: {ml_summary['total_bets']}")
print(f"Record: {ml_summary['wins']}-{ml_summary['losses']}")
print(f"Win %: {ml_summary['win_pct']:.1%}")
print(f"Total profit: ${ml_summary['total_profit']:+,.0f}")
print(f"\nProfit by year:")
for year, profit in sorted(ml_summary['profit_by_year'].items()):
    print(f"  {year}: ${profit:+,.0f}")

In [ ]:
# Visualize cumulative profit
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cumulative profit over time (all bets)
dog_sim['cum_profit'] = dog_sim['profit'].cumsum()
axes[0].plot(range(len(dog_sim)), dog_sim['cum_profit'], linewidth=1.5)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Bet Number')
axes[0].set_ylabel('Cumulative Profit ($)')
axes[0].set_title('Underdog ATS: Cumulative Profit Over Time')
axes[0].fill_between(range(len(dog_sim)), dog_sim['cum_profit'], 0,
                      where=dog_sim['cum_profit'] > 0, alpha=0.3, color='green')
axes[0].fill_between(range(len(dog_sim)), dog_sim['cum_profit'], 0,
                      where=dog_sim['cum_profit'] <= 0, alpha=0.3, color='red')

# Profit by round
round_profit = dog_sim.groupby('round_name')['profit'].sum().reindex(
    ['First Round', 'Second Round', 'Sweet 16', 'Elite 8', 'Final Four', 'Championship'])
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in round_profit]
axes[1].bar(round_profit.index, round_profit.values, color=colors)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Total Profit ($)')
axes[1].set_title('Underdog ATS Profit by Round')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('data/profitability.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. 2026 TOURNAMENT PREVIEW

Based on historical trends, here are the key edges to watch for the 2026 tournament:

In [ ]:
print("2026 MARCH MADNESS BETTING CHEAT SHEET")
print("=" * 60)

# First Round spread insights
r1 = df[df['round_num'] == 1]
print(f"\n--- FIRST ROUND ---")
print(f"Overall favorite ATS rate: {r1['favorite_covered'].mean():.1%}")

# Best first-round underdog matchups
r1_matchups = r1.groupby('matchup').agg(
    games=('favorite_covered', 'count'),
    dog_covers=('favorite_covered', lambda x: (~x).sum()),
).reset_index()
r1_matchups['dog_cover_pct'] = r1_matchups['dog_covers'] / r1_matchups['games']
r1_matchups = r1_matchups[r1_matchups['games'] >= 5].sort_values('dog_cover_pct', ascending=False)

print(f"\nBest first-round underdog ATS matchups:")
for _, row in r1_matchups.head(5).iterrows():
    print(f"  {row['matchup']}: Dogs cover {row['dog_cover_pct']:.0%} ({int(row['dog_covers'])}/{int(row['games'])})")

# Totals insights
print(f"\n--- TOTALS ---")
print(f"First Round avg total: {r1['total_points'].mean():.1f}")
print(f"First Round median total: {r1['total_points'].median():.1f}")
over_150 = (r1['total_points'] > 150).mean()
under_130 = (r1['total_points'] < 130).mean()
print(f"First Round over 150: {over_150:.1%}")
print(f"First Round under 130: {under_130:.1%}")

# Sweet 16+ totals drop
late = df[df['round_num'] >= 3]
print(f"\nSweet 16+ avg total: {late['total_points'].mean():.1f}")
print(f"Sweet 16+ under 130: {(late['total_points'] < 130).mean():.1%}")

# Upset probability by first round matchup
print(f"\n--- MONEYLINE UPSETS ---")
r1_upset = r1.groupby('matchup').agg(
    games=('upset', 'count'),
    upsets=('upset', 'sum'),
).reset_index()
r1_upset['upset_pct'] = r1_upset['upsets'] / r1_upset['games']
r1_upset = r1_upset[r1_upset['games'] >= 5].sort_values('upset_pct', ascending=False)

print("First-round upset rates (higher seeds winning):")
for _, row in r1_upset.iterrows():
    print(f"  {row['matchup']}: {row['upset_pct']:.0%} ({int(row['upsets'])}/{int(row['games'])})")

print(f"\n--- STRATEGY RECOMMENDATIONS ---")
print(f"1. Target underdogs ATS in matchups where dogs cover >52.4%")
print(f"2. Lean UNDER in Sweet 16+ games (scoring drops significantly)")
print(f"3. Sprinkle 12-seed ML bets vs 5-seeds (historically profitable upset rate)")
print(f"4. Fade big favorites laying 15+ points (they rarely cover)")
print(f"5. Watch for 8v9 unders — tight, evenly-matched games")

In [ ]:
# Heatmap: Seed matchup performance matrix
fig, ax = plt.subplots(figsize=(12, 10))

# Create a pivot of upset rates by seed pairing
r1_games = df[df['round_num'] == 1].copy()
pivot_data = r1_games.groupby(['favorite_seed', 'underdog_seed']).agg(
    upset_rate=('upset', 'mean'),
    games=('upset', 'count'),
).reset_index()

# Only show matchups with enough data
pivot_data = pivot_data[pivot_data['games'] >= 3]
pivot = pivot_data.pivot(index='favorite_seed', columns='underdog_seed', values='upset_rate')

sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', center=0.3,
            ax=ax, linewidths=0.5, vmin=0, vmax=0.6)
ax.set_title('First Round Upset Rate by Seed Pairing\n(Favorite Seed vs Underdog Seed)')
ax.set_xlabel('Underdog Seed')
ax.set_ylabel('Favorite Seed')

plt.tight_layout()
plt.savefig('data/upset_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAnalysis complete! Check the 'data/' folder for exported charts.")